In [2]:
from google.colab import drive; drive.mount('/content/drive')
import sys, importlib, numpy as np, pandas as pd, torch
from scipy.stats import wilcoxon
from transformers import AutoTokenizer, AutoModelForSequenceClassification
CODE = "/content/drive/MyDrive/esci/code"; sys.path.insert(0, CODE)
import metrics, cross_encoder as C, train_cross_encoder as T
for m in (metrics, C, T): importlib.reload(m)
from metrics import evaluate
BASE = "/content/drive/MyDrive/esci"

df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
qs = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_queries.parquet")[["query_id", "rule", "scope_group"]]
val = df[df.part == "val"].merge(qs, on="query_id", how="left").reset_index(drop=True); del df
CAT = {"mobile_computing": "Mobile & Computing", "components_accessories": "Components & Accessories",
       "home_electronics": "Home Electronics", "imaging": "Cameras & Imaging", "audio": "Audio",
       "computer_peripherals": "Computer Peripherals", "gaming": "Gaming", "smart_home": "Smart Home"}
val["category"] = np.where(val["rule"] == "A", val["scope_group"].map(CAT), None)
qtext = [C.query_to_text(q, category=c) for q, c in zip(val["query"], val["category"])]

recs = val.to_dict("records")
def first_words(t, n): return " ".join(C._clean(t).split()[:n])
def short_plus(r):                              # short title + every other field M2 might give
    parts = [first_words(r["product_title"], 4)]
    b, c, f = C._clean(r["product_brand"]), C._clean(r["product_color"]), C._clean(r["product_bullet_point"])
    if b: parts.append(f"brand: {b}")
    if c: parts.append(f"color: {c}")
    if f: parts.append("features: " + f[:200])
    return " | ".join(parts)
full  = np.array([C.product_to_text(C.esci_row_to_product(r), "full") for r in recs], dtype=object)
short = np.array([first_words(r["product_title"], 4) for r in recs], dtype=object)
splus = np.array([short_plus(r) for r in recs], dtype=object)
val["is_short"] = np.random.default_rng(11).random(len(val)) < 0.5      # fixed random half of candidates
SETUPS = {"all_full": full, "all_short": short, "all_short_plus": splus,
          "mixed_short": np.where(val.is_short, short, full),
          "mixed_short_plus": np.where(val.is_short, splus, full)}
print(pd.DataFrame({"full": full[:2], "short": short[:2], "short_plus": splus[:2]}).to_string())

def best(log_file, prefix):                     # best epoch on NORMAL formats only
    L = pd.read_csv(f"{BASE}/results/{log_file}"); L = L[L["style"].isin(C.STYLES)]
    L = L[L.run.str.fullmatch(rf"{prefix}__q_focused_cat(__s\d+)?")]
    ep = L.groupby(["run", "epoch"])["ndcg@10"].mean().reset_index()
    return ep.loc[ep.groupby("run")["ndcg@10"].idxmax()][["run", "epoch"]]
MODELS = {"mix": best("s7_grid_val_log.csv", "minilm_l6"), "aug": best("s9_aug_val_log.csv", "minilm_l6_aug")}
for g, b in MODELS.items(): print(g, b.values.tolist())

def bias(v):   # >0 means Exact products with SHORT text sit lower in the list than Exact products with FULL text
    v = v.copy(); v["pct"] = v.groupby("query_id")["score"].rank(ascending=False, pct=True)
    g = v[v.gain == 1].groupby(["query_id", "is_short"])["pct"].mean().unstack().dropna()
    return g[True] - g[False]

NDCG, BIAS, rows = {}, {}, []
for g, b in MODELS.items():
    for _, r in b.iterrows():
        path = f"{BASE}/models/{r.run}/epoch_{int(r.epoch)}"
        tok = AutoTokenizer.from_pretrained(path)
        m = AutoModelForSequenceClassification.from_pretrained(path).to("cuda").eval()
        for s, txt in SETUPS.items():
            v = val[["query_id", "gain", "is_short"]].copy()
            v["score"] = T.score_pairs(m, tok, qtext, list(txt), 256, 128)
            mtr, pq = evaluate(v); rows.append({"model": g, "run": r.run, "setup": s, **mtr})
            NDCG[(g, s)] = NDCG.get((g, s), []) + [pq.set_index("query_id")["ndcg@10"]]
            if s.startswith("mixed"): BIAS[(g, s)] = BIAS.get((g, s), []) + [bias(v)]
        del m; torch.cuda.empty_cache(); print("scored", r.run)

avg = lambda d, k: pd.concat(d[k], axis=1).mean(axis=1)
R = pd.DataFrame(rows)
print("\nNDCG@10 (mean of 3 seeds)\n", R.groupby(["setup", "model"])["ndcg@10"].mean().unstack().reindex(list(SETUPS)).round(4).to_string())

def p_of(x): x = x[x != 0]; return wilcoxon(x).pvalue if len(x) else float("nan")
print("\nR1 recovery (all_short_plus - all_short):")
for g in MODELS:
    d = avg(NDCG, (g, "all_short_plus")) - avg(NDCG, (g, "all_short"))
    print(f"  {g}: {d.mean():+.4f} | p = {p_of(d):.4f}")
print("\nR2 bias (Exact items: short minus full, in list-percentile; +0.10 = 10% of the list lower):")
for g in MODELS:
    for s in ["mixed_short", "mixed_short_plus"]:
        bq = avg(BIAS, (g, s)); print(f"  {g:3s} {s:16s}: {bq.mean():+.4f} | queries {len(bq)} | p = {p_of(bq):.4f}")
for s in ["mixed_short", "mixed_short_plus"]:
    d = avg(BIAS, ("aug", s)) - avg(BIAS, ("mix", s)); d = d.dropna()
    print(f"R3 aug - mix bias [{s}]: {d.mean():+.4f} | p = {p_of(d):.4f}")
R.to_csv(f"{BASE}/results/s9d_completeness.csv", index=False)

Mounted at /content/drive
                                                                                                                                                                                                             full                               short                                                                                                                                                                                                                                                                                      short_plus
0                                                                                        PANASONIC Amplified Cordless Phone with Digital Answering Machine - KX-TGM450S - 1 Handset (Silver) | Panasonic | Silver  PANASONIC Amplified Cordless Phone      PANASONIC Amplified Cordless Phone | brand: Panasonic | color: Silver | features: Hear and be heard, loud and clear with volume boost control for Amplified Caller voice volume up to 50 db.,

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6__q_focused_cat


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6__q_focused_cat__s43


...

scored minilm_l6_aug__q_focused_cat


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6_aug__q_focused_cat__s43


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6_aug__q_focused_cat__s44

NDCG@10 (mean of 3 seeds)
 model                aug     mix
setup                           
all_full          0.9273  0.9278
all_short         0.8780  0.8811
all_short_plus    0.8998  0.8998
mixed_short       0.8968  0.8966
mixed_short_plus  0.9067  0.9073

R1 recovery (all_short_plus - all_short):
  mix: +0.0187 | p = 0.0000
  aug: +0.0218 | p = 0.0000

R2 bias (Exact items: short minus full, in list-percentile; +0.10 = 10% of the list lower):
  mix mixed_short     : +0.2679 | queries 596 | p = 0.0000
  mix mixed_short_plus: +0.1842 | queries 596 | p = 0.0000
  aug mixed_short     : +0.1549 | queries 596 | p = 0.0000
  aug mixed_short_plus: +0.1880 | queries 596 | p = 0.0000
R3 aug - mix bias [mixed_short]: -0.1130 | p = 0.0000
R3 aug - mix bias [mixed_short_plus]: +0.0039 | p = 0.2162


In [3]:
import importlib.util, pandas as pd
CODE, BASE = "/content/drive/MyDrive/esci/code", "/content/drive/MyDrive/esci"
def load(name, f):
    s = importlib.util.spec_from_file_location(name, f"{CODE}/{f}"); m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
old, new = load("ce_old", "cross_encoder_v_s6.py"), load("ce_new", "cross_encoder.py")
qs = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_queries.parquet")
ch = [(q, old.query_to_text(q), new.query_to_text(q)) for q in qs["query"] if old.query_to_text(q) != new.query_to_text(q)]
print(len(ch), "of", len(qs), "ESCI queries change")
for c in ch: print(c)

0 of 5665 ESCI queries change
